# Final Analysis Notebook  
**Tracking subjective symptom improvement from patient narratives in Mobile Health: an observational Natural Language Processing study**

This notebook reproduces the core analytical workflow described in the manuscript, including:

1. Data loading  
2. Text preprocessing  
3. Construction of the keyword-derived perceived improvement indicator  
4. Sentiment scoring  
5. TF-IDF feature extraction  
6. Satisfaction regression analysis  
7. Classification with Logistic Regression  
8. Non-linear benchmarking with Random Forest  
9. Evaluation tables for manuscript reporting  

> Replace file paths and column names where needed to match your working dataset.

In [ ]:
# Core imports
import re
import os
import json
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    mean_squared_error,
    r2_score
)

# Optional
try:
    import statsmodels.api as sm
    STATSMODELS_AVAILABLE = True
except Exception:
    STATSMODELS_AVAILABLE = False

print("Libraries loaded.")

## 1. Load dataset

Expected core columns:
- `review_text`: raw or cleaned review text
- `satisfaction_rating`: numeric satisfaction score
- optional metadata columns such as date, platform, or review ID

If your dataset already contains a cleaned text column, you can map it below.

In [ ]:
# === UPDATE THIS PATH ===
DATA_PATH = "data/your_dataset.csv"

df = pd.read_csv(DATA_PATH)

# === UPDATE THESE COLUMN NAMES IF NEEDED ===
TEXT_COL = "review_text"
SAT_COL = "satisfaction_rating"

print(df.head())
print(df.columns.tolist())
print(df.shape)

## 2. Basic cleaning and preparation

In [ ]:
# Keep required columns
df = df.dropna(subset=[TEXT_COL, SAT_COL]).copy()

# Cast types
df[TEXT_COL] = df[TEXT_COL].astype(str)
df[SAT_COL] = pd.to_numeric(df[SAT_COL], errors="coerce")
df = df.dropna(subset=[SAT_COL]).copy()

# Simple text cleaning function
def simple_clean(text: str) -> str:
    text = text.lower().strip()
    text = re.sub(r"\s+", " ", text)
    return text

df["clean_text"] = df[TEXT_COL].apply(simple_clean)

print(df[[TEXT_COL, "clean_text", SAT_COL]].head())
print(df.shape)

## 3. Construct keyword-derived perceived improvement indicator

This corresponds to the manuscript's **keyword-derived perceived improvement indicator**.  
Replace the keyword list below with the exact final lexicon used in your study.

In [ ]:
# === UPDATE THIS KEYWORD LIST TO MATCH YOUR MANUSCRIPT ===
improvement_keywords = [
    "relief", "pain free", "pain-free", "better", "improved",
    "recovered", "recovery", "no pain", "less pain", "much better",
    "symptom improved", "feeling better"
]

pattern = re.compile("|".join(re.escape(k) for k in improvement_keywords), flags=re.IGNORECASE)

df["improvement_indicator"] = df["clean_text"].apply(lambda x: 1 if pattern.search(x) else 0)

print(df["improvement_indicator"].value_counts(dropna=False))
print("Positive rate:", round(df["improvement_indicator"].mean(), 4))

## 4. Sentiment scoring

If you used a specific sentiment tool or lexicon in the manuscript, replicate that here.  
The placeholder below uses a simple lexicon-based score for demonstration only. Replace with your actual scoring approach.

In [ ]:
# Placeholder sentiment lexicons (replace with your actual approach)
positive_words = {"good", "great", "helpful", "better", "excellent", "relief", "effective", "satisfied"}
negative_words = {"bad", "worse", "pain", "poor", "ineffective", "unsatisfied", "slow", "expensive"}

def simple_sentiment_score(text: str) -> float:
    tokens = re.findall(r"\b\w+\b", text.lower())
    if not tokens:
        return 0.0
    pos = sum(1 for t in tokens if t in positive_words)
    neg = sum(1 for t in tokens if t in negative_words)
    return (pos - neg) / max(len(tokens), 1)

df["sentiment_score"] = df["clean_text"].apply(simple_sentiment_score)

df[["clean_text", "sentiment_score"]].head()

## 5. Descriptive statistics

In [ ]:
desc = {
    "n_reviews": len(df),
    "mean_satisfaction": df[SAT_COL].mean(),
    "sd_satisfaction": df[SAT_COL].std(),
    "improvement_rate": df["improvement_indicator"].mean(),
    "mean_sentiment": df["sentiment_score"].mean(),
    "sd_sentiment": df["sentiment_score"].std()
}
pd.DataFrame([desc]).round(3)

## 6. Regression analysis: sentiment and satisfaction

This section estimates the association between sentiment polarity and satisfaction ratings.

In [ ]:
X_reg = df[["sentiment_score"]].copy()
y_reg = df[SAT_COL].copy()

linreg = LinearRegression()
linreg.fit(X_reg, y_reg)
y_pred_reg = linreg.predict(X_reg)

reg_results = {
    "coefficient_sentiment": linreg.coef_[0],
    "intercept": linreg.intercept_,
    "rmse": mean_squared_error(y_reg, y_pred_reg, squared=False),
    "r2": r2_score(y_reg, y_pred_reg)
}

pd.DataFrame([reg_results]).round(3)

In [ ]:
# Optional statsmodels output for manuscript reporting
if STATSMODELS_AVAILABLE:
    X_sm = sm.add_constant(X_reg)
    model_sm = sm.OLS(y_reg, X_sm).fit()
    print(model_sm.summary())
else:
    print("statsmodels not installed; using sklearn summary only.")

## 7. TF-IDF feature extraction for classification

In [ ]:
X_text = df["clean_text"]
y = df["improvement_indicator"]

X_train_text, X_test_text, y_train, y_test = train_test_split(
    X_text,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

tfidf = TfidfVectorizer(
    max_features=1000,
    ngram_range=(1, 1),
    min_df=2
)

X_train_tfidf = tfidf.fit_transform(X_train_text)
X_test_tfidf = tfidf.transform(X_test_text)

print("Train TF-IDF shape:", X_train_tfidf.shape)
print("Test TF-IDF shape:", X_test_tfidf.shape)

## 8. Logistic Regression classifier

In [ ]:
log_reg = LogisticRegression(max_iter=1000, random_state=42)
log_reg.fit(X_train_tfidf, y_train)
y_pred_lr = log_reg.predict(X_test_tfidf)

## 9. Random Forest benchmark

In [ ]:
rf = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    n_jobs=-1
)
rf.fit(X_train_tfidf, y_train)
y_pred_rf = rf.predict(X_test_tfidf)

## 10. Evaluation helper

In [ ]:
def evaluate_model(y_true, y_pred, model_name="Model"):
    results = {
        "Model": model_name,
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "F1-score": f1_score(y_true, y_pred, zero_division=0)
    }
    print(f"=== {model_name} ===")
    print("Accuracy :", round(results["Accuracy"], 4))
    print("Precision:", round(results["Precision"], 4))
    print("Recall   :", round(results["Recall"], 4))
    print("F1-score :", round(results["F1-score"], 4))
    print("\nClassification Report:")
    print(classification_report(y_true, y_pred, zero_division=0))
    print("Confusion Matrix:")
    print(confusion_matrix(y_true, y_pred))
    return results

## 11. Run classifier evaluation

In [ ]:
lr_results = evaluate_model(y_test, y_pred_lr, "Logistic Regression (TF-IDF)")
rf_results = evaluate_model(y_test, y_pred_rf, "Random Forest (TF-IDF)")

## 12. Model comparison table for manuscript

In [ ]:
results_df = pd.DataFrame([lr_results, rf_results])
for col in ["Accuracy", "Precision", "Recall", "F1-score"]:
    results_df[col] = results_df[col].round(3)

results_df

## 13. Optional: top TF-IDF features from logistic regression

In [ ]:
feature_names = np.array(tfidf.get_feature_names_out())
coefs = log_reg.coef_[0]

top_pos_idx = np.argsort(coefs)[-20:][::-1]
top_neg_idx = np.argsort(coefs)[:20]

top_positive_features = pd.DataFrame({
    "feature": feature_names[top_pos_idx],
    "coefficient": coefs[top_pos_idx]
})

top_negative_features = pd.DataFrame({
    "feature": feature_names[top_neg_idx],
    "coefficient": coefs[top_neg_idx]
})

print("Top positive features")
display(top_positive_features)

print("Top negative features")
display(top_negative_features)

## 14. Optional: Random Forest feature importances

In [ ]:
rf_importances = rf.feature_importances_
top_rf_idx = np.argsort(rf_importances)[-20:][::-1]

rf_top_features = pd.DataFrame({
    "feature": feature_names[top_rf_idx],
    "importance": rf_importances[top_rf_idx]
})

rf_top_features

## 15. Export outputs

In [ ]:
os.makedirs("outputs", exist_ok=True)

results_df.to_csv("outputs/model_comparison_results.csv", index=False)

if 'rf_top_features' in globals():
    rf_top_features.to_csv("outputs/random_forest_top_features.csv", index=False)

if 'top_positive_features' in globals():
    top_positive_features.to_csv("outputs/logreg_top_positive_features.csv", index=False)
    top_negative_features.to_csv("outputs/logreg_top_negative_features.csv", index=False)

print("Outputs saved in the outputs/ folder.")

## 16. Reporting notes for the manuscript

Use the following outputs in the revised paper:

- Descriptive statistics from Section 5
- Regression coefficient and R² from Section 6
- Classification performance table from Section 12
- Optional feature interpretation from Sections 13–14

Make sure the final terminology in the manuscript is consistent with the revised reviewer responses:
- **keyword-derived perceived improvement indicator**
- **satisfaction rating**
- **sentiment polarity**